# Project: AI Newsletter Pipeline

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app).

You will build a **Flow** that runs three **crews**—research, writing, assembly—with **`@start`**, **`@listen`**, **`@router`**, Pydantic **state**, **structured output**, **task context**, **memory**, and **`output_file`**.

In [ ]:
!pip install -q crewai crewai-tools pydantic

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Step 1: State model and structured-output schemas

`NewsletterState` is the Flow’s shared Pydantic state. `ResearchBrief` and `QualityRating` back `output_pydantic` on tasks.

In [ ]:
from typing import Optional

from pydantic import BaseModel, Field

from crewai.flow.flow import Flow, listen, router, start


class NewsletterState(BaseModel):
    topic: str = "AI Agents"
    research: str = ""
    articles: list[str] = []
    newsletter: str = ""
    quality_score: int = 0


class ResearchBrief(BaseModel):
    summary: str = Field(description="Executive summary of the research")
    key_points: list[str] = Field(description="Bullet-level trends")


class QualityRating(BaseModel):
    score: int = Field(ge=1, le=10, description="Editorial readiness 1-10")
    note: str = ""

## Step 2: Research crew (researcher + analyst, 2 tasks)

The analyst task uses **`context=[research_task]`** so it runs after the researcher. **`output_pydantic=ResearchBrief`** on the analyst task yields structured briefing text for the writer.

In [ ]:
from crewai import Agent, Crew, Process, Task

researcher = Agent(
    role="AI Trends Researcher",
    goal="Find timely AI trends and credible angles for a weekly newsletter",
    backstory="You scan news, papers, and product launches for what practitioners care about this week.",
    verbose=True,
)

analyst = Agent(
    role="Research Analyst",
    goal="Turn raw research into a tight briefing the writer can use",
    backstory="You compress noise into clear themes and bullets.",
    verbose=True,
)

research_task = Task(
    description=(
        "Research the weekly AI landscape for the topic: {topic}. "
        "Return notable trends, tools, and one-line evidence for each."
    ),
    expected_output="Structured bullet research suitable for an analyst.",
    agent=researcher,
)

analysis_task = Task(
    description=(
        "Using the research above, produce a briefing: summary + key_points "
        "for a newsletter writer. Topic: {topic}."
    ),
    expected_output="A ResearchBrief JSON-matching object.",
    agent=analyst,
    context=[research_task],
    output_pydantic=ResearchBrief,
)

research_crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

## Step 3: Writing crew (one writer, three sections + quality task)

Sections 2 and 3 use **`context`** on the previous section task. A final task outputs **`QualityRating`** for the **`@router`** gate.

In [ ]:
writer = Agent(
    role="Newsletter Writer",
    goal="Write three engaging Markdown sections for busy AI readers",
    backstory="You write clear, scannable sections with strong ledes.",
    verbose=True,
)

section_1 = Task(
    description=(
        "Write **Section 1** (intro + lead story) for topic {topic}. "
        "Base it on this research briefing:\n{research}"
    ),
    expected_output="Markdown section 1 only.",
    agent=writer,
)

section_2 = Task(
    description=(
        "Write **Section 2** (second story or trend) continuing the newsletter. "
        "Stay consistent with Section 1."
    ),
    expected_output="Markdown section 2 only.",
    agent=writer,
    context=[section_1],
)

section_3 = Task(
    description=(
        "Write **Section 3** (roundup / links / what to watch). "
        "Align tone with prior sections."
    ),
    expected_output="Markdown section 3 only.",
    agent=writer,
    context=[section_2],
)

rate_task = Task(
    description="Rate editorial readiness of the three sections combined (1-10) and one short note.",
    expected_output="QualityRating",
    agent=writer,
    context=[section_1, section_2, section_3],
    output_pydantic=QualityRating,
)

writing_crew = Crew(
    agents=[writer],
    tasks=[section_1, section_2, section_3, rate_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

## Step 4: Assembly crew (editor + `output_file`)

The editor merges **`{combined}`** (all sections joined in the Flow) into one Markdown file via **`output_file`**.

In [ ]:
editor = Agent(
    role="Newsletter Editor",
    goal="Assemble sections into one polished Markdown newsletter file",
    backstory="You unify voice, fix transitions, and enforce a simple house style.",
    verbose=True,
)

assemble_task = Task(
    description=(
        "Combine the following draft sections into one newsletter in Markdown. "
        "Add a title line and a date placeholder. Unify voice and transitions.\n\n"
        "{combined}\n\nTopic: {topic}."
    ),
    expected_output="Full newsletter Markdown",
    agent=editor,
    output_file="newsletter.md",
)

assembly_crew = Crew(
    agents=[editor],
    tasks=[assemble_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

## Step 5–6: Flow (`@start` → research → writing → `@router` → publish / revise) and `kickoff`

**`quality_gate`** returns **`publish`** or **`revise`**. The revise branch lightly edits **`articles`**, bumps the score, then runs the same assembly crew. Both paths pass **`combined`** into **`assembly_crew.kickoff`**.

In [ ]:
class NewsletterFlow(Flow[NewsletterState]):
    @start()
    def set_topic(self):
        self.state.topic = self.state.topic or "AI Agents"
        return self.state.topic

    @listen(set_topic)
    def run_research_crew(self, topic: str):
        out = research_crew.kickoff(inputs={"topic": topic})
        brief: Optional[ResearchBrief] = None
        for o in out.tasks_output:
            p = getattr(o, "pydantic", None)
            if isinstance(p, ResearchBrief):
                brief = p
                break
        if brief is not None:
            self.state.research = brief.summary + "\n" + "\n".join(f"- {x}" for x in brief.key_points)
        else:
            self.state.research = out.raw
        return self.state.research

    @listen(run_research_crew)
    def run_writing_crew(self, research: str):
        wout = writing_crew.kickoff(inputs={"topic": self.state.topic, "research": research})
        bodies = []
        rating: Optional[QualityRating] = None
        for o in wout.tasks_output:
            p = getattr(o, "pydantic", None)
            if isinstance(p, QualityRating):
                rating = p
            else:
                raw = getattr(o, "raw", None) or str(o)
                bodies.append(raw)
        if len(bodies) >= 3:
            self.state.articles = bodies[:3]
        else:
            self.state.articles = bodies
        if rating is not None:
            self.state.quality_score = rating.score
        else:
            self.state.quality_score = min(10, sum(len(a) for a in self.state.articles) // 400)
        return wout.raw

    @router(run_writing_crew)
    def quality_gate(self, _):
        if self.state.quality_score >= 7:
            return "publish"
        return "revise"

    @listen("publish")
    def publish_path(self):
        combined = "\n\n---\n\n".join(self.state.articles)
        aout = assembly_crew.kickoff(inputs={"topic": self.state.topic, "combined": combined})
        self.state.newsletter = aout.raw
        return self.state.newsletter

    @listen("revise")
    def revise_path(self):
        self.state.articles = [
            (a + "\n\n*( lightly edited for clarity )*") for a in self.state.articles
        ]
        self.state.quality_score = max(self.state.quality_score, 8)
        combined = "\n\n---\n\n".join(self.state.articles)
        aout = assembly_crew.kickoff(inputs={"topic": self.state.topic, "combined": combined})
        self.state.newsletter = aout.raw
        return self.state.newsletter


flow = NewsletterFlow()
print(flow.kickoff())
print(flow.state.model_dump())

## What you learned

- **Flows** coordinate **multiple crews** with **`Flow[NewsletterState]`** and decorators **`@start`**, **`@listen`**, **`@router`**.
- **Task `context`** chains work inside each crew; **`combined` + `kickoff(inputs=...)`** avoids reusing task objects across crews when assembling.
- **`output_pydantic`** gives typed **`ResearchBrief`** and **`QualityRating`** outputs; **`output_file`** persists the final **`newsletter.md`**.
- **`memory=True`** on crews helps agents retain useful context within the run.
- A **quality router** lets you branch to **`publish`** vs **`revise`** before assembly (here, revise still finishes by running the assembly crew).